<a href="https://colab.research.google.com/github/prakash587/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prakash587/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I chose the Refresh / Content Opportunity Scoring lane. This is primarily a ranking/scoring problem because the goal is not simply to predict whether a page is declining, but to rank pages by priority so content editors know which pages should be reviewed first. Each page receives a score based on multiple observable signals such as content age, impressions, CTR, engagement, and average position. The output is a prioritized review queue that supports decisions like refreshing, expanding, monitoring, or pruning content. A ranking approach is more useful than a simple yes/no prediction because editors need to know which pages deserve attention first.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target of this project is a content refresh opportunity score, which ranks pages based on how strongly they should be considered for review. In the starter dataset, I will use is_declining_label as a proxy target to evaluate the model. This label is defined by a rule, where a page is marked as declining if trend_direction == "down".

Because this label is derived from the current data rather than measured from a future outcome, it is a proxy, not a true observed target. In a stronger version of this project, I would use an observed future outcome, such as whether a page declines or recovers during a later time window, to avoid leakage and better reflect real-world performance. Therefore, the current work should be viewed as decision-support rather than a definitive prediction of future performance.

In [4]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

df[["trend_direction", "is_declining_label"]].head(10)

,trend_direction,is_declining_label
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The main success metric for this project is Precision@50. This measures the proportion of the top 50 pages ranked by the model that are actually declining according to the proxy label (is_declining_label). I chose this metric because content teams typically review only a limited number of pages, so it is more important that the highest-ranked recommendations are accurate than achieving high overall accuracy. A good model should achieve a higher Precision@50 than a simple hand-written rule while still producing interpretable recommendations that support content refresh decisions.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
from sklearn.tree import DecisionTreeClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()


features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features].replace([float("inf"), float("-inf")], 0).fillna(0)
y = df["is_declining_label"]

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

scores = tree.predict_proba(X)[:, 1]

precision50 = precision_at_k(scores, y, 50)

print(f"Precision@50: {precision50:.3f}")

Precision@50: 0.600


In [8]:
print("Total pages:", len(df))
print("Declining pages:", df["is_declining_label"].sum())

Total pages: 30000
Declining pages: 16262


n my experiment, the model achieved a Precision@50 of 0.600, meaning that 60% of the top 50 recommended pages were correctly identified as declining. This makes Precision@50 an appropriate metric for evaluating a ranked content refresh queue.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis in this project is one content page. Each row in the dataset represents a single content page with its observed metrics, including impressions, CTR, average position, engagement rate, content age, and trend direction. These features are used to evaluate whether a page should be prioritized for refresh. The output will support a ranked list of pages for content review and refresh.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load the starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Features used for my lane
lane_df = df[
    [
        "content_id",
        "impressions_90d",
        "content_age_days",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "engagement_rate",
        "trend_direction"
    ]
]

print("Unit of analysis: One row = One content page")
print("Number of pages:", len(lane_df))

lane_df.head()

Unit of analysis: One row = One content page
Number of pages: 30000


,content_id,impressions_90d,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,trend_direction
0,content_304f48230142,3803,187,20,0.76,10.6,5.88,down
1,content_a1fb4e703a9e,15320,445,25,0.05,20.3,0.00,down
2,content_9aa793d4d895,12581,141,20,0.09,36.5,0.00,down
3,content_331d6c4de07b,11751,463,22,0.49,6.2,1.28,stable
4,content_d99b7a2d90ca,19140,263,14,0.13,44.0,0.00,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule, such as selecting pages that are old and have high impressions, considers only a few conditions and cannot capture more complex relationships between multiple features. Whether a page should be refreshed may depend on a combination of content age, impressions, CTR, average position, engagement rate, and other signals. Machine learning can learn these patterns from the data and assign a more accurate priority score than a single if-statement.A simple if-statement uses only one or two conditions and cannot capture the complex relationships between multiple features. The model supports decision-making by ranking pages for review, while still requiring human judgment before any content changes are made.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Compare the hand-written rule with the decision tree model

# Define a simple hand-written rule score
df["hand_rule_score"] = df["content_age_days"] * df["impressions_90d"]

hand_precision = precision_at_k(df["hand_rule_score"], y, 50)

tree_precision = precision_at_k(
    tree.predict_proba(X)[:, 1],
    y,
    50
)

print(f"Hand Rule Precision@50 : {hand_precision:.3f}")
print(f"ML Model Precision@50  : {tree_precision:.3f}")

Hand Rule Precision@50 : 0.440
ML Model Precision@50  : 0.600


I compared a simple fixed rule based on content age and impressions with the decision tree model. The fixed rule achieved a Precision@50 of 0.440, while the ML model achieved 0.600. This suggests that a simple rule cannot capture the interactions between multiple features, whereas the ML model can learn more complex patterns from the data. The model is intended to support content refresh decisions, not replace human judgment.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.